---
# Curso: Data Mining — Semana 11 - Lunes
## <font color=green>Laboratorio: evaluación de clusters y clientes nuevos</font>

**Docente:** Soledad Espezúa Llerena

---

## <font color=green>Caso</font>

Una entidad financiera quiere **segmentar a sus clientes** para diseñar campañas y políticas de crédito diferenciadas.

Tiene datos de 850 clientes y sabe si cada cliente cayó en mora (`moroso`), pero 150 clientes todavía no tienen historial de pago.

Formaremos los segmentos **sin usar** `moroso` y lo usaremos al final para describir el riesgo de cada segmento.

## <font color=green>Objetivos</font>

- evaluar K-means con **K creciente** usando inercia, Silhouette y Davies-Bouldin;
- revisar el Silhouette de cada cliente y reconocer los valores negativos;
- evaluar DBSCAN con **eps creciente** observando el número de clusters y el ruido;
- perfilar los segmentos y describir su riesgo con la variable `moroso`;
- asignar clientes nuevos con los modelos ya ajustados, sin volver a ajustarlos.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score

# Configuración visual
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

# <font color=blue>PARTE I. Datos</font>

## <font color=green>1. Cargamos los datos</font>

Una fila = un cliente. Variables:

- `id_cliente`: identificador del cliente;
- `edad`: edad en años;
- `nivel_educativo`: de 1 (menor) a 5 (mayor);
- `anios_empleo`: años en el empleo actual;
- `ingreso`: ingreso en miles de US;
- `deuda_tarjeta` y `otras_deudas`: deudas en miles de US$;
- `ratio_deuda_ingreso`: deuda total / ingreso × 100;
- `codigo_direccion`: código de la zona de residencia;
- `moroso`: 1 = incumplió pagos, 0 = no incumplió, vacío = sin historial.

In [ ]:
df = pd.read_csv("Clientes_Segmentacion.csv")

print("Filas y columnas:", df.shape)

display(df.head())

In [ ]:
# Revisamos cuántos valores faltan en cada columna.
display(df.isna().sum().to_frame("faltantes"))

## <font color=green>2. Clientes con historial y clientes nuevos</font>

Los 150 clientes con `moroso` vacío todavía no tienen historial de pago.

No los eliminamos: formaremos los segmentos con los **700 clientes con historial** y, al final, trataremos a los 150 como **clientes nuevos**.

In [ ]:
# notnull() es True cuando tiene dato; isna() es True cuando está vacía.
historial = df[df["moroso"].notnull()].copy()   # clientes con historial
nuevos = df[df["moroso"].isna()].copy()       # clientes sin historial

print("Clientes con historial:", historial.shape[0])
print("Clientes nuevos:", nuevos.shape[0])

## <font color=green>3. Elegimos los atributos y estandarizamos</font>

- `id_cliente` y `codigo_direccion` son códigos: no tienen sentido en una distancia.
- `moroso` es el resultado que queremos describir: **no entra** al clustering.

K-means y DBSCAN usan distancias, por eso estandarizamos los atributos.

In [ ]:
# Atributos que usaremos para calcular distancias entre clientes.
atributos = [
    "edad",
    "nivel_educativo",
    "anios_empleo",
    "ingreso",
    "deuda_tarjeta",
    "otras_deudas",
    "ratio_deuda_ingreso"
]

X = historial[atributos]

# Ajustamos el escalador SOLO con los clientes con historial.
# Más adelante usaremos este mismo escalador con los clientes nuevos.
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

print("Dimensión de X_std:", X_std.shape)

 <font color=red>**Nota:** la variable `moroso` **no participa** en el clustering ni en las métricas internas. La usaremos para **seleccionar** atributos y, al final, para describir el riesgo de cada segmento.</font>

 `moroso` es el resultado que queremos describir; si entrara a la distancia, los segmentos se armarían con la respuesta y ya no podríamos usarla para evaluar si los segmentos sirven.

El escalador se ajusta con los clientes con historial porque con ellos se forman los segmentos. A los clientes nuevos se les aplica la **misma** transformación (`transform`), para que queden en la misma escala que los clientes con los que se formaron los segmentos.

# <font color=blue>PARTE II. K-means con K creciente</font>

Recordatorio:

- **inercia**: siempre baja al aumentar K; buscamos el codo;
- **Silhouette**: entre −1 y 1; más alto es mejor;
- **Davies-Bouldin**: mayor o igual que 0; más bajo es mejor.

## <font color=green>4. Probamos K = 2, 3, …, 8</font>

Además de las métricas, guardamos dos datos que ayudan a decidir al banco:

- el tamaño del segmento más pequeño (**¿el cluster es lo bastante grande para usarlo?**)--> Por ejm.a un segmento de 7 u 11 clientes no se le diseña una campaña ni una política de crédito. Además es inestable: con otra semilla o con clientes nuevos puede desaparecer o cambiar.
- la diferencia de morosidad entre el segmento más riesgoso y el menos riesgoso (**¿los segmentos tienen riesgos distintos?**). La usaremos para medir si los segmentos le sirven al banco, que es algo que las métricas internas no miden.
   *   Es la morosidad del segmento más riesgoso menos la del menos riesgoso.
   *   Si es cercana a 0, todos los segmentos tienen el mismo riesgo y la segmentación no le sirve al banco.


In [ ]:
resultados_kmeans = []

for k in range(2, 9):

    modelo = KMeans(
        n_clusters=k,       # número de clusters
        init="k-means++",
        n_init=10,          # prueba 10 inicializaciones y se queda con la mejor
        random_state=42
    )

    labels = modelo.fit_predict(X_std)

    # Tasa de morosidad de cada segmento (proporción de morosos).
    morosidad = historial.groupby(labels)["moroso"].mean()

    # Número de clientes en cada segmento.
    tamanos = pd.Series(labels).value_counts()

    resultados_kmeans.append({
        "K": k,
        "inercia": modelo.inertia_,
        "silhouette": silhouette_score(X_std, labels),
        "davies_bouldin": davies_bouldin_score(X_std, labels),
        "segmento_mas_pequeno": tamanos.min(),
        "dif_morosidad": morosidad.max() - morosidad.min()
    })

metricas_kmeans = pd.DataFrame(resultados_kmeans)

display(metricas_kmeans.round(3))

### ¿Para qué sirven las dos últimas columnas?

Inercia, Silhouette y Davies-Bouldin dicen si los grupos son compactos y separados, pero no si le sirven al banco. Para eso guardamos dos columnas más:

- `segmento_mas_pequeno`: número de clientes del segmento más chico. Por ejemplo, con K = 2 los 700 clientes se reparten en dos grupos de 545 y 155, y se guarda 155. Al aumentar K los grupos se dividen y este número baja. Un segmento con muy pocos clientes no sirve para diseñar una campaña y suele ser inestable.
- `dif_morosidad`: morosidad del segmento más riesgoso menos la del menos riesgoso. Si es cercana a 0, todos los segmentos tienen el mismo riesgo y la segmentación no ayuda a diferenciar las políticas de crédito.

Las dos columnas se leen juntas: `dif_morosidad` suele crecer al aumentar K, pero a veces solo porque aparecen segmentos muy pequeños con tasas extremas (por ejemplo, un grupo de 7 clientes todos morosos). Un buen K separa el riesgo **y** tiene segmentos de tamaño razonable.

`moroso` no participa en el clustering: solo lo usamos para evaluar si los segmentos son útiles.

## <font color=green>5. Graficamos las métricas</font>

In [ ]:
# Inercia: buscamos el codo, donde deja de bajar bruscamente.
ax = sns.lineplot(data=metricas_kmeans, x="K", y="inercia", marker="o")
ax.set(title="Inercia según K", xlabel="K", ylabel="Inercia")
plt.show()

# Silhouette: buscamos el valor más alto.
ax = sns.lineplot(data=metricas_kmeans, x="K", y="silhouette", marker="o")
ax.set(title="Silhouette según K", xlabel="K", ylabel="Silhouette")
plt.show()

# Davies-Bouldin: buscamos el valor más bajo.
ax = sns.lineplot(data=metricas_kmeans, x="K", y="davies_bouldin", marker="o")
ax.set(title="Davies-Bouldin según K", xlabel="K", ylabel="Davies-Bouldin")
plt.show()

## Pregunta

¿Qué K sugiere cada métrica? ¿Coinciden? ¿Qué K elegirías para el banco? Revisa también el tamaño del segmento más pequeño y la diferencia de morosidad.

## Respuesta esperada

Si miramos solo las métricas, no coinciden:

- La **inercia** baja de forma continua y no muestra un codo claro.
- **Silhouette** es máximo en **K = 2** (0.375), pero esos dos grupos tienen casi la misma morosidad (diferencia de 0.029): no ayudan al banco a distinguir riesgos.
- **Davies-Bouldin** es mínimo en **K = 5** (1.241).

Si descartamos K = 2, las métricas sí coinciden en **K = 5**: ahí Silhouette tiene su segundo valor más alto (0.246, un máximo local) y Davies-Bouldin su mínimo (1.241).

Elegimos **K = 5**: el segmento más pequeño tiene 43 clientes y la morosidad tiene diferencia de 0.457. Con K ≥ 6 aparecen segmentos muy pequeños (12, 11 y 7 clientes).

Las siluetas son bajas (alrededor de 0.25): los segmentos se solapan, así que se describen como tendencias y no como grupos totalmente separados.

## <font color=green>6. Ajustamos K-means con el K elegido</font>

In [ ]:
K_ELEGIDO = 5

kmeans = KMeans(
    n_clusters=K_ELEGIDO,
    init="k-means++",
    n_init=10,
    random_state=42
)

# Guardamos el segmento de cada cliente en la tabla historial.
historial["segmento"] = kmeans.fit_predict(X_std)

print("Silhouette:", round(silhouette_score(X_std, historial["segmento"]), 3))
print("Davies-Bouldin:", round(davies_bouldin_score(X_std, historial["segmento"]), 3))

# Número de clientes en cada segmento.
display(historial["segmento"].value_counts().sort_index().to_frame("n"))

In [ ]:
# Para visualizar, resumimos los 7 atributos estandarizados en 2 componentes con PCA.
# PCA solo sirve para dibujar: K-means se ajustó con los 7 atributos.
pca = PCA(n_components=2)
Z = pca.fit_transform(X_std)

historial["Z1"] = Z[:, 0]
historial["Z2"] = Z[:, 1]

ax = sns.scatterplot(
    data=historial,
    x="Z1",
    y="Z2",
    hue=historial["segmento"].astype("category"),   # cada segmento con un color
    s=40
)

ax.set(
    title=f"K-means (K = {K_ELEGIDO}) en el plano de PCA",
    xlabel="Componente 1",
    ylabel="Componente 2"
)
ax.legend(title="Segmento")
plt.show()

## <font color=green>7. Silhouette de cada cliente</font>

`silhouette_samples` calcula s(i) para cada cliente. El Silhouette score es el promedio de esos valores.

Un s(i) negativo indica que el cliente está, en promedio, más cerca de otro segmento que del suyo.

In [ ]:
# s(i) de cada cliente, en el mismo orden que X_std.
historial["silhouette"] = silhouette_samples(X_std, historial["segmento"])

promedio = historial["silhouette"].mean()
print("Promedio de s(i) = Silhouette score:", round(promedio, 3))

# Distribución de s(i) en cada segmento.
ax = sns.boxplot(data=historial, x="segmento", y="silhouette")

ax.axhline(0, color="black", linewidth=1)                        # línea en s = 0
ax.axhline(promedio, color="red", linestyle="--", linewidth=1)   # promedio general

ax.set(title="Silhouette de cada cliente por segmento", xlabel="Segmento", ylabel="s(i)")
plt.show()

In [ ]:
K = 5
fig, ax = plt.subplots(figsize=(7, 6))
y_inf = 10
for c in range(K):
    valores = np.sort(historial.loc[historial["segmento"] == c, "silhouette"].values)
    y_sup = y_inf + len(valores)
    ax.fill_betweenx(np.arange(y_inf, y_sup), 0, valores, alpha=0.8, label=f"Segmento {c} (n={len(valores)})")
    y_inf = y_sup + 10
ax.axvline(promedio, color="red", ls="--", label=f"Promedio = {promedio:.3f}")
ax.set(title=f"Diagrama de silueta — K-means, K={K}", xlabel="Coeficiente de silueta", ylabel="Clientes", yticks=[])
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

display(historial.groupby("segmento")["silhouette"].agg(["mean", lambda s: (s < 0).sum()])
        .rename(columns={"mean": "silueta_media", "<lambda_0>": "n_negativos"}).round(3))

Los clientes con Silhouette negativo, indican que ese segmento está en promedio más cerca de otro segmento que del suyo

## Pregunta

¿Qué segmento tiene el Silhouette medio más bajo y más clientes con s(i) negativo? ¿Qué significa eso?

## Respuesta esperada

* El **segmento 4** tiene el Silhouette medio más bajo (0.005) y 23 de sus 43 clientes con s(i) negativo, más de la mitad. Esos clientes están, en promedio, más cerca de otro segmento que del suyo: el segmento 4 no está bien separado de los demás.

* El segmento 0 tiene 18 negativos de 150 clientes y el segmento 3, 13 de 105.
* Los segmentos 1 y 2 no tienen negativos; el 1 es el más compacto (0.354).

Por eso sirve el gráfico por segmento: muestra cuáles están bien formados y cuáles no.

# <font color=blue>PARTE III. DBSCAN con eps creciente</font>

Con DBSCAN lo que cambia el resultado es el radio `eps`.

Probamos varios valores de `eps` y observamos qué pasa.

## <font color=green>8. Gráfico de k-distancias</font>

Fijamos `min_samples = 8`, un poco más que el número de atributos (7).

Buscamos la zona donde la curva empieza a subir rápido: es un valor inicial para `eps`.

In [ ]:
min_samples = 8

# Para cada cliente buscamos sus 8 vecinos más cercanos.
vecinos = NearestNeighbors(n_neighbors=min_samples)
vecinos.fit(X_std)

distancias, _ = vecinos.kneighbors(X_std)

# Tomamos la distancia al último vecino y la ordenamos de menor a mayor.
k_distancias = np.sort(distancias[:, -1])

kdist_df = pd.DataFrame({
    "cliente_ordenado": range(len(k_distancias)),
    "k_distancia": k_distancias
})

ax = sns.lineplot(data=kdist_df, x="cliente_ordenado", y="k_distancia")
ax.set(
    title=f"Gráfico de {min_samples}-distancias",
    xlabel="Clientes ordenados",
    ylabel=f"Distancia al vecino {min_samples}"
)
plt.show()

## <font color=green>9. Probamos eps creciente</font>

Probamos valores en la zona donde sube la curva (de 1.8 a 4) y dos valores menores (1.0 y 1.5) para comparar.

Para cada `eps` guardamos el número de clusters, el porcentaje de ruido y las métricas.

Silhouette y Davies-Bouldin se calculan **sin el ruido** y solo cuando hay al menos 2 clusters.

In [ ]:
comparacion_eps = []

for eps in [1.0, 1.5, 1.8, 2.0, 2.5, 3.0, 3.5, 4.0]:

    modelo = DBSCAN(
        eps=eps,                  # radio de la vecindad
        min_samples=min_samples   # mínimo de puntos para formar una zona densa
    )

    labels = modelo.fit_predict(X_std)

    # Cuenta cuántos clusters encontró DBSCAN, sin contar el ruido (-1).
    etiquetas = set(labels)
    n_clusters = len(etiquetas) - (1 if -1 in etiquetas else 0)

    # Porcentaje de clientes marcados como ruido.
    porcentaje_ruido = np.mean(labels == -1) * 100

    # Las métricas se calculan sin el ruido y necesitan al menos 2 clusters.
    sin_ruido = labels != -1

    if n_clusters >= 2:
        sil = silhouette_score(X_std[sin_ruido], labels[sin_ruido])
        db = davies_bouldin_score(X_std[sin_ruido], labels[sin_ruido])
    else:
        sil = np.nan
        db = np.nan

    comparacion_eps.append({
        "eps": eps,
        "n_clusters": n_clusters,
        "ruido_%": porcentaje_ruido,
        "silhouette_sin_ruido": sil,
        "davies_bouldin_sin_ruido": db
    })

comparacion_eps = pd.DataFrame(comparacion_eps)

display(comparacion_eps.round(3))

Con estos clientes, **DBSCAN** no encuentra grupos: en toda la zona razonable de `eps` sale un solo cluster más algunos puntos de ruido.

In [ ]:
# Número de clusters según eps.
ax = sns.lineplot(data=comparacion_eps, x="eps", y="n_clusters", marker="o")
ax.set(title="DBSCAN: clusters encontrados según eps", xlabel="eps", ylabel="Número de clusters")
plt.show()

# Porcentaje de ruido según eps.
ax = sns.lineplot(data=comparacion_eps, x="eps", y="ruido_%", marker="o")
ax.set(title="DBSCAN: porcentaje de ruido según eps", xlabel="eps", ylabel="% de ruido")
plt.show()

### Interpretación

- **Clusters según eps:** con `eps = 1.0` aparecen 4 clusters, pero desde `eps = 1.5` hasta 4.0 hay **un solo cluster**. No existe un valor de `eps` con el que DBSCAN encuentre segmentos en estos clientes.
- **Ruido según eps:** hasta `eps = 1.8` el ruido baja rápido (de 33.4 % a 4.6 %). Después baja de a poco (de 3.6 % a 1.1 %): agrandar el radio solo absorbe a unos pocos clientes extremos.

Elegimos `eps = 1.8` porque ahí empieza a subir la curva de k-distancias y ahí el ruido deja de bajar bruscamente.

> Regla práctica: el gráfico de k-distancias propone un rango; con **eps creciente** buscamos el tramo donde el número de clusters es estable y el ruido deja de bajar bruscamente, y elegimos el `eps` más chico de ese tramo.

### Probamos entonces combinaciones de `min_samples` y `eps`.

In [ ]:
# Probamos varias combinaciones de min_samples y eps.
combinaciones = []

for ms in [3, 5, 8, 10, 15, 20]:
    for eps in [0.75, 1.0, 1.25, 1.5, 2.0, 3.0]:

        labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_std)

        # Número de clusters sin contar el ruido (-1).
        etiquetas = set(labels)
        n_clusters = len(etiquetas) - (1 if -1 in etiquetas else 0)

        combinaciones.append({
            "min_samples": ms,
            "eps": eps,
            "n_clusters": n_clusters,
            "ruido_%": np.mean(labels == -1) * 100
        })

combinaciones = pd.DataFrame(combinaciones)

# Tabla de doble entrada: filas = min_samples, columnas = eps.
tabla_clusters = combinaciones.pivot(index="min_samples", columns="eps", values="n_clusters")
tabla_ruido = combinaciones.pivot(index="min_samples", columns="eps", values="ruido_%")

sns.heatmap(tabla_clusters, annot=True, fmt=".0f", cmap="Blues")
plt.title("Número de clusters según min_samples y eps")
plt.show()

sns.heatmap(tabla_ruido, annot=True, fmt=".0f", cmap="Oranges")
plt.title("Porcentaje de ruido según min_samples y eps")
plt.show()

Donde hay varios clusters (arriba a la izquierda) el ruido es alto, y donde el ruido es bajo hay un solo cluster.

Con min_samples = 3 y eps = 1.0, DBSCAN sí forma clusters, pero cada uno corresponde a un nivel educativo: un cluster tiene solo clientes de nivel 1, otro solo de nivel 2, y así. En lo demás (edad, ingreso, deudas) los grupos son casi iguales.

Esto pasa porque nivel_educativo es una variable ordinal discreta: solo toma los valores 1 a 5. Al estandarizarla, entre un nivel y el siguiente hay una distancia de 1.08. Con un radio menor (eps = 1.0), DBSCAN no puede pasar de un nivel al siguiente y cada nivel queda separado. Con un radio apenas mayor (eps = 1.25), los grupos se unen otra vez en uno solo.

## <font color=green>10. DBSCAN con eps = 1.8</font>

In [ ]:
dbscan = DBSCAN(
    eps=1.8,
    min_samples=3
)

historial["dbscan"] = dbscan.fit_predict(X_std)

# La etiqueta -1 corresponde a ruido.
display(historial["dbscan"].value_counts().sort_index().to_frame("n"))

In [ ]:
# Comparamos el perfil promedio del ruido con el del resto de clientes.
historial["es_ruido"] = historial["dbscan"] == -1

display(historial.groupby("es_ruido")[atributos].mean().round(1))

In [ ]:
# Mismo plano de PCA, ahora coloreado según DBSCAN.
# Convertimos las etiquetas a texto y llamamos "ruido" a la etiqueta -1.
historial["grupo_dbscan"] = historial["dbscan"].astype(str).replace({"-1": "ruido"})

ax = sns.scatterplot(
    data=historial,
    x="Z1",
    y="Z2",
    hue="grupo_dbscan",     # color según cluster o ruido
    style="grupo_dbscan",   # forma distinta para el ruido
    s=40
)

ax.set(
    title="DBSCAN (eps = 1.8) en el plano de PCA",
    xlabel="Componente 1",
    ylabel="Componente 2"
)
ax.legend(title="DBSCAN")
plt.show()

In [ ]:
# Las dos variables donde el ruido más se diferencia del resto (ver la tabla anterior).
ax = sns.scatterplot(
    data=historial,
    x="ingreso",
    y="otras_deudas", #"nivel_educativo",
    hue="grupo_dbscan",     # color según cluster o ruido
    style="grupo_dbscan",   # forma distinta para el ruido
    s=40
)

ax.set(
    title="DBSCAN (eps = 1.8) según ingreso y nivel educativo",
    xlabel="Ingreso (miles de US$)",
    ylabel="otras_deudas"
)
ax.legend(title="DBSCAN")
plt.show()

In [ ]:
# moroso no participó en DBSCAN: solo lo usamos para describir el resultado.
# Tasa de morosidad del ruido y de la región densa.
display(historial.groupby("grupo_dbscan")["moroso"].mean().round(2).to_frame("morosidad"))

## Pregunta

Según la tabla anterior, ¿qué caracteriza a los clientes que DBSCAN marca como ruido? ¿Para qué podría servir DBSCAN en este caso?

## Respuesta esperada

Los 26 clientes marcados como ruido son mayores (43 años frente a 34), tienen más años de empleo (16 frente a 8), ingresos mucho más altos (152.5 frente a 41.5), deudas unas 6 veces mayores y un ratio deuda/ingreso más alto (18.1 frente a 10): son clientes **atípicos de altos ingresos y alta deuda**.

Como vimos con eps creciente, DBSCAN no encuentra segmentos en estos clientes (un solo cluster desde `eps = 1.5`). Aquí sirve para **detectar clientes atípicos**, que podrían pasar a evaluación manual.

# <font color=blue>PARTE IV. Perfil de los segmentos de K-means</font>

El perfil se calcula con los valores **originales** (sin escalar), para leerlo en años, miles de US$ y porcentajes.

Aquí sí usamos `moroso`: para describir el riesgo de cada segmento.

## <font color=green>11. Perfil promedio y morosidad</font>

In [ ]:
# Promedio de cada atributo por segmento, en unidades originales.
perfil = historial.groupby("segmento")[atributos].mean().round(1)

# Agregamos el tamaño y la tasa de morosidad de cada segmento.
perfil["n"] = historial.groupby("segmento").size()
perfil["morosidad"] = historial.groupby("segmento")["moroso"].mean().round(2)

display(perfil)

In [ ]:
# Tasa de morosidad por segmento (promedio de moroso).
ax = sns.barplot(data=historial, x="segmento", y="moroso", errorbar=None)
ax.set(title="Tasa de morosidad por segmento", xlabel="Segmento", ylabel="Proporción de morosos")
plt.show()

In [ ]:
# Para comparar atributos con escalas distintas, expresamos cada promedio
# en desviaciones estándar respecto del promedio general (valor z).
perfil_z = (perfil[atributos] - X.mean()) / X.std()

plt.figure(figsize=(10, 3))
sns.heatmap(perfil_z, annot=True, fmt=".2f", center=0, cmap="vlag")
plt.title("Perfil relativo de los segmentos (valores z)")
plt.show()

**Interpretación**
* Segmento 0 · Establecidos de bajo riesgo
* Segmento 1 · Jóvenes de ingreso bajo
* Segmento 2 · Jóvenes con estudios superiores
* Segmento 3 · Sobreendeudados
* Segmento 4 · Altos ingresos y alta deuda

# <font color=blue>PARTE V. Llegan clientes nuevos</font>

Los modelos ya están ajustados con los 700 clientes con historial. A un cliente nuevo **no** se le vuelve a ajustar nada:

- escalador: `transform` (nunca `fit_transform`);
- K-means: `predict`, que le asigna el centroide más cercano;
- DBSCAN: **no tiene** `predict`.

## <font color=green>12. Error frecuente: volver a ajustar el escalador</font>

In [ ]:
# Un solicitante nuevo.
solicitante = pd.DataFrame({
    "edad": [28],
    "nivel_educativo": [2],
    "anios_empleo": [3],
    "ingreso": [35],
    "deuda_tarjeta": [2.5],
    "otras_deudas": [4.0]
})

# Calculamos el ratio igual que en los datos: deuda total / ingreso × 100.
solicitante["ratio_deuda_ingreso"] = (
    100 * (solicitante["deuda_tarjeta"] + solicitante["otras_deudas"]) / solicitante["ingreso"]
).round(1)

display(solicitante)

In [ ]:
# MAL: fit_transform calcula la media y la desviación con una sola fila.
print("MAL :", StandardScaler().fit_transform(solicitante[atributos]).round(2))

# BIEN: transform usa la media y la desviación de los clientes con historial.
print("BIEN:", scaler.transform(solicitante[atributos]).round(2))

Con `fit_transform` sobre una sola fila, la media es el propio cliente y todo queda en 0: el cliente parece promedio en todo.

## <font color=green>13. Asignamos al solicitante con K-means</font>

In [ ]:
# Escalamos con el escalador ya ajustado.
x_solicitante = scaler.transform(solicitante[atributos])

# predict asigna el segmento del centroide más cercano.
seg = kmeans.predict(x_solicitante)[0]

print("Segmento asignado:", seg)
print("Distancia a cada centroide:", kmeans.transform(x_solicitante).round(2))
print("Morosidad histórica del segmento:", perfil.loc[seg, "morosidad"])

## <font color=green>14. Asignamos un segmento a los 150 clientes nuevos</font>

In [ ]:
# Escalamos a los nuevos con el MISMO escalador.
X_nuevos_std = scaler.transform(nuevos[atributos])

# Asignamos un segmento a cada cliente nuevo.
nuevos["segmento"] = kmeans.predict(X_nuevos_std)

display(nuevos["segmento"].value_counts().sort_index().to_frame("n_nuevos"))

## <font color=green>15. ¿Qué tan lejos queda cada cliente nuevo de su segmento?</font>

Aunque un cliente sea muy distinto de todos, `predict` le asigna el centroide más cercano.

Para detectar esos casos comparamos su distancia al centroide con la de los clientes con historial.

In [ ]:
# kmeans.transform da la distancia a cada centroide; min(axis=1) toma la más cercana.
dist_historial = kmeans.transform(X_std).min(axis=1)

# Umbral: el 95 % de los clientes con historial está más cerca que esto de su centroide.
umbral = np.percentile(dist_historial, 95)

# Distancia de cada cliente nuevo a su centroide.
nuevos["dist_centroide"] = kmeans.transform(X_nuevos_std).min(axis=1)
nuevos["revisar"] = nuevos["dist_centroide"] > umbral

print("Umbral (percentil 95):", round(umbral, 2))
print("Clientes nuevos para revisión manual:", nuevos["revisar"].sum())

display(nuevos[nuevos["revisar"]][atributos + ["segmento", "dist_centroide"]].round(2))

## <font color=green>16. DBSCAN no tiene `predict`</font>

Una regla sencilla: si el cliente nuevo está a una distancia menor o igual que `eps` de algún **punto núcleo**, toma el cluster de ese núcleo; si no, es ruido.

In [ ]:
# Puntos núcleo de DBSCAN y el cluster de cada uno.
nucleos = X_std[dbscan.core_sample_indices_]
cluster_nucleos = dbscan.labels_[dbscan.core_sample_indices_]

# Para cada cliente nuevo buscamos el núcleo más cercano.
vecino = NearestNeighbors(n_neighbors=1)
vecino.fit(nucleos)

dist_nucleo, idx_nucleo = vecino.kneighbors(X_nuevos_std)

# Si está a una distancia <= eps, toma el cluster del núcleo; si no, queda como ruido (-1).
nuevos["dbscan"] = np.where(
    dist_nucleo[:, 0] <= dbscan.eps,
    cluster_nucleos[idx_nucleo[:, 0]],
    -1
)

display(nuevos["dbscan"].value_counts().sort_index().to_frame("n_nuevos"))

## Pregunta

¿Por qué a los clientes nuevos se les aplica `transform` y no `fit_transform`? ¿Qué harías con los clientes nuevos que superan el umbral de distancia o que DBSCAN marca como ruido?

## Respuesta

Porque el escalador debe aplicar la **misma** media y desviación con que se formaron los segmentos. Con `fit_transform` sobre los nuevos la escala cambia (con una sola fila todo queda en 0) y el cliente se compara con centroides que están en otra escala.

K-means siempre asigna un segmento, incluso a clientes muy distintos. Los 14 nuevos que superan el umbral (2.94) reciben un segmento, pero conviene revisarlos manualmente antes de aplicarles la política de ese segmento. DBSCAN deja a 13 nuevos como ruido: también son candidatos a evaluación manual.

Si llegan muchos casos así, hay que volver a entrenar los modelos y a perfilar los segmentos.

## Ejercicio

Llega un solicitante con 50 años, nivel educativo 3, 20 años de empleo, ingreso de 150, deuda de tarjeta de 12 y otras deudas de 18 (miles de US$).

1. Calcula su `ratio_deuda_ingreso`.
2. Asígnale un segmento con K-means.
3. ¿Supera el umbral de distancia? ¿Lo enviarías a revisión manual?

In [ ]:
cliente = pd.DataFrame({
    "edad": [50],
    "nivel_educativo": [3],
    "anios_empleo": [20],
    "ingreso": [150],
    "deuda_tarjeta": [12],
    "otras_deudas": [18]
})

# 1. Ratio deuda/ingreso.
cliente["ratio_deuda_ingreso"] = (
    100 * (cliente["deuda_tarjeta"] + cliente["otras_deudas"]) / cliente["ingreso"]
).round(1)

# 2. Escalamos con el escalador ajustado y asignamos el segmento.
x_cliente = scaler.transform(cliente[atributos])
seg_cliente = kmeans.predict(x_cliente)[0]

# 3. Distancia a su centroide y comparación con el umbral.
dist_cliente = kmeans.transform(x_cliente).min()

print("Ratio deuda/ingreso:", cliente.loc[0, "ratio_deuda_ingreso"])
print("Segmento asignado:", seg_cliente, "| morosidad histórica:", perfil.loc[seg_cliente, "morosidad"])
print("Distancia a su centroide:", round(dist_cliente, 2), "| umbral:", round(umbral, 2))
print("¿Revisión manual?", dist_cliente > umbral)

## Respuesta esperada

El ratio es 30 / 150 × 100 = **20.0**. K-means lo asigna al **segmento 4** (altos ingresos y alta deuda, morosidad histórica de 42 %).

Su distancia al centroide (3.36) supera el umbral (2.94): es un cliente poco típico incluso para su segmento, así que conviene enviarlo a **revisión manual**.